In [ ]:
import numpy as np
from pathlib import Path
from generator import systemInit, setup_ham_rse, calculate_spectral_density
from tqdm.auto import tqdm
import sisl
import h5py
import gc

import ipywidgets as widgets
from ipywidgets import interact, Dropdown, fixed, IntSlider

In [2]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.set_theme(style="white", context="notebook")

# 2. Fine-tune specific font sizes and behaviors
plt.rcParams.update({
    "font.family": "serif",        # Use serif for a formal/academic look
    "font.size": 9,               # Base font size
    "axes.titlesize": 18,          # Subplot titles (your .set(title=...) calls)
    "axes.labelsize": 14,          # X and Y axis labels
    "xtick.labelsize": 10,         # Size of numbers on X-axis
    "ytick.labelsize": 10,         # Size of numbers on Y-axis
    "legend.fontsize": 14,         # Legend text
    "figure.titlesize": 20,        # Overall figure title
    "axes.labelpad": 10,           # Distance between label and axis
    "mathtext.fontset": "cm",      # Computer Modern (LaTeX look for math)
    "savefig.dpi": 300             # High-res exports
})

REAL_PART_CMP = sns.color_palette("RdBu", as_cmap=True)

In [3]:
OUTDIR = Path("conv_data")
if not OUTDIR.exists():
    print("Err: dir '{}' does NOT exist".format(OUTDIR))
    CWD = Path.cwd()
    print('Create \'{}\' at \'{}\' '.format(OUTDIR, CWD))
    # OUT_DIR.mkdir(parents=False, exist_ok=False)
    # print('####\'{}\' created'.format(OUT_DIR))
else:
    print("dir '{}' exists!".format(OUTDIR))

dir 'conv_data' exists!


In [4]:
a = np.array([5., 1.])
b = np.array([1, 3, 4, 5])*(-1)
ETAS = np.multiply.outer(10.0**b, a, dtype=np.float128).ravel()
print('etas ({}) : {}'.format(len(ETAS), ETAS))

dE = 0.1
EMAX = 0.4
EMIN = -EMAX
ENERGIES = np.arange(EMIN, EMAX+dE/2, dE, dtype=np.float128).round(1)
print('E ({})    : {}'.format(len(ENERGIES), ENERGIES))

E0_IDX = np.argwhere(ENERGIES == 0).ravel() # returns tuple of arrays of len == ndim(ENERGIES)
E0_IDX = E0_IDX[0] # since ENERGIES is 1d array, the tuple has len 1
print('idx(E=0) : {}'.format(E0_IDX))

etas (8) : [5.e-01 1.e-01 5.e-03 1.e-03 5.e-04 1.e-04 5.e-05 1.e-05]
E (9)    : [-0.4 -0.3 -0.2 -0.1 -0.   0.1  0.2  0.3  0.4]
idx(E=0) : 4


In [ ]:
NLIST = np.array([4*i+1 for i in range(2, 7)])[::-1]
print(f"Ns : {NLIST}, shape={NLIST.shape}")

NK1 = int(np.ceil(3*900/12))
SEMI_AXIS = 0 # semi-infinite axis
K_AXES = 1 # k-sampling axis/axes

with h5py.File(OUTDIR / 'RSE_data.h5', 'w') as file:
    file.attrs['E'] = ENERGIES
    file.attrs['E0_idx'] = E0_IDX
    file.attrs['ETAS'] = ETAS
    
    for i, N in enumerate(tqdm(NLIST, desc="Looping tiling", leave=True)):
        
        group_N = file.create_group(f"N_{N}")
        
        Ham0 = systemInit(1.43, -2.7)
        HamNN : sisl.Hamiltonian = Ham0.tile(N, 0).tile(N, 1)
        HamNN.set_nsc([1,1,1])
        
        rse = sisl.RealSpaceSE(Ham0, SEMI_AXIS, K_AXES, (N, N, 1))
        rse.setup(eta=ETAS[0],
                bz=sisl.MonkhorstPack(Ham0, [1, NK1, 1]))
        _, elec_idx = rse.real_space_coupling(ret_indices=True)
        all_atoms = np.arange(0, HamNN.na)
        device_atoms = np.delete(all_atoms, elec_idx)
        atoms_idx = np.concatenate([elec_idx, device_atoms])
        HamNN_re = HamNN.sub(atoms_idx)
        HamNN_re.reduce()
        
        del HamNN
        
        num_atoms = len(HamNN_re)
        num_E = len(ENERGIES)
        RSEs_shape = (num_E, num_atoms, num_atoms)
        
        
        group_N.create_dataset("xyz", data=HamNN_re.geometry.xyz)
        group_N.create_dataset("atoms_idx", data=atoms_idx)
        group_N.create_dataset("elec_idx", data=elec_idx)
        

        for j, eta in enumerate(tqdm(ETAS, desc="Looping etas", leave=False)):
            if (j > 0):
                rse = sisl.RealSpaceSE(Ham0, SEMI_AXIS, K_AXES, (N, N, 1))
                rse.setup(eta=eta,
                        bz=sisl.MonkhorstPack(Ham0, [1, NK1, 1]))
            # _, elec_idx = rse.real_space_coupling()
            # if j == 0:
            #     all_atoms = np.arange(0, HamNN.na)
            #     device_atoms = np.delete(all_atoms, elec_idx)
            #     atoms_idx = np.concatenate([elec_idx, device_atoms])
            #     HamNN_re = HamNN.sub(atoms_idx)
            #     HamNN_re.reduce()
            #     group_N.create_dataset("xyz", data=HamNN_re.geometry.xyz)
            #     group_N.create_dataset("atoms_idx", data=atoms_idx)
            #     group_N.create_dataset("elec_idx", data=elec_idx)
            
            
            
            RSEs_group_eta = group_N.create_dataset(f"eta_{eta}",shape=RSEs_shape,
                                          dtype=np.complex128, compression='gzip')            
        
            # H_mat = HamNN_re.Hk(format="array", dtype=np.complex128)
            # S_mat = HamNN_re.Sk(format="array", dtype=np.complex128)
            for i, E in enumerate(tqdm(ENERGIES, desc='Looping energies', leave=False)):
                z = E + eta*1j
                RSE = rse.self_energy(z)
                RSE_re = RSE[atoms_idx, :][:, atoms_idx]
                RSEs_group_eta[i, :, :] = RSE_re


Ns : [25 21 17 13  9], shape=(5,)


Looping tiling:   0%|          | 0/5 [00:00<?, ?it/s]

Looping etas:   0%|          | 0/8 [00:00<?, ?it/s]

Looping energies:   0%|          | 0/9 [00:00<?, ?it/s]

Looping energies:   0%|          | 0/9 [00:00<?, ?it/s]

...

Looping energies:   0%|          | 0/9 [00:00<?, ?it/s]

Looping energies:   0%|          | 0/9 [00:00<?, ?it/s]

Looping energies:   0%|          | 0/9 [00:00<?, ?it/s]

Looping energies:   0%|          | 0/9 [00:00<?, ?it/s]

: 